可換性×

In [1]:
import random
import numpy as np
import sys
from scipy.sparse import csr_matrix, hstack, vstack

In [2]:
L=12
l_h = L // 2
J=3
P=768
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

In [3]:
class GirthOptimizer:
    def __init__(self, P=768, L_half=6):
        self.P = P
        self.L_half = L_half
        self.mid = P // 2
        # 基底の生成
        self.rho_A = self._generate_random_cycle(range(0, self.mid))
        self.rho_B = self._generate_random_cycle(range(self.mid, self.P))

    def _generate_random_cycle(self, r):
        indices = list(r)
        random.shuffle(indices)
        p = list(range(self.P))
        for i in range(len(indices)):
            p[indices[i]] = indices[(i + 1) % len(indices)]
        return tuple(p)

    def compose(self, p1, p2):
        return tuple(p1[p2[i]] for i in range(self.P))

    def get_power(self, base_p, k):
        res = tuple(range(self.P))
        curr = base_p
        k %= self.mid
        while k > 0:
            if k % 2 == 1: res = self.compose(res, curr)
            curr = self.compose(curr, curr)
            k //= 2
        return res

    def count_4_cycles_block(self, current_blocks):
        """現在のブロック集合において発生している長さ4のサイクルを高速計算"""
        if not current_blocks: return 0
        # ブロックを疎行列化して結合
        H = vstack([tuple_to_sparse(p, self.P) for p in current_blocks])
        H_int = H.astype(np.int64)
        Gram = (H_int @ H_int.T).toarray()
        np.fill_diagonal(Gram, 0)
        return np.sum(Gram * (Gram - 1)) // 4

    def search_optimized(self):
        print(f"--- Optimized Search Starting (P={self.P}) ---")
        
        # 1. Fブロックの逐次決定
        F = []
        for i in range(self.L_half):
            best_f = None
            min_c4 = float('inf')
            
            # 探索範囲を限定（候補から最適な指数を探す）
            sample_indices = random.sample(range(1, self.mid), min(50, self.mid-1))
            for k in sample_indices:
                candidate_f = self.get_power(self.rho_A, k)
                # 暫定的に追加してサイクルを計測
                test_c4 = self.count_4_cycles_block(F + [candidate_f])
                if test_c4 < min_c4:
                    min_c4 = test_c4
                    best_f = candidate_f
                if min_c4 == 0: break # ガース条件を満たせば即採用
            
            F.append(best_f)
            print(f"F[{i}] determined. Current C4 cycles in F-set: {min_c4}")

        # 2. Gブロックの逐次決定 (Fとの直交性を保ちつつ)
        G = []
        for j in range(self.L_half):
            best_g = None
            min_c4 = float('inf')
            
            sample_indices = random.sample(range(1, self.mid), min(50, self.mid-1))
            for k in sample_indices:
                candidate_g = self.get_power(self.rho_B, k)
                # 全体の行列としての接続を確認（FとGの組み合わせ）
                test_c4 = self.count_4_cycles_block(F + G + [candidate_g])
                if test_c4 < min_c4:
                    min_c4 = test_c4
                    best_g = candidate_g
                if min_c4 == 0: break
                
            G.append(best_g)
            print(f"G[{j}] determined. Current C4 cycles in Total-set: {min_c4}")

        return F, G

def tuple_to_sparse(p, size):
    rows = np.arange(size)
    cols = np.array(p)
    return csr_matrix((np.ones(size, dtype=np.int8), (rows, cols)), shape=(size, size))

In [4]:
class HighEntropyAPM:
    def __init__(self, P=768, L_half=6):
        self.P = P
        self.L_half = L_half
        self.mid = P // 2
        
        # 前半と後半で独立したランダムサイクルを生成
        self.rho_A = self._generate_random_cycle(range(0, self.mid))
        self.rho_B = self._generate_random_cycle(range(self.mid, self.P))

    def _generate_random_cycle(self, r):
        indices = list(r)
        if len(indices) < 2:
            return tuple(range(self.P))
        random.shuffle(indices)
        p = list(range(self.P))
        for i in range(len(indices)):
            p[indices[i]] = indices[(i + 1) % len(indices)]
        return tuple(p)

    def compose(self, p1, p2):
        return tuple(p1[p2[i]] for i in range(self.P))

    def get_power(self, base_p, k):
        res = tuple(range(self.P))
        curr = base_p
        if self.mid <= 0:
            return res
        k %= self.mid
        while k > 0:
            if k % 2 == 1: res = self.compose(res, curr)
            curr = self.compose(curr, curr)
            k //= 2
        return res

    def inject_swap(self, p, r):
        p_list = list(p)
        indices = list(r)
        if len(indices) < 2:
            return p
        i1, i2 = random.sample(indices, 2)
        p_list[i1], p_list[i2] = p_list[i2], p_list[i1]
        return tuple(p_list)

    def construct(self):
        # ValueError を防ぐため、人口サイズをチェック (range(1, mid) の長さは mid-1)
        pop_size = self.mid - 1
        
        if pop_size >= self.L_half:
            # 重複なしで安全にサンプリング
            f_indices = random.sample(range(1, self.mid), self.L_half)
            g_indices = random.sample(range(1, self.mid), self.L_half)
        else:
            # Pが極端に小さい場合は重複を許容して randint を使用
            limit = max(1, self.mid - 1)
            f_indices = [random.randint(1, limit) for _ in range(self.L_half)]
            g_indices = [random.randint(1, limit) for _ in range(self.L_half)]
        
        F = [self.get_power(self.rho_A, k) for k in f_indices]
        G = [self.get_power(self.rho_B, k) for k in g_indices]

        # 非可換性の注入 (Disjoint Support)
        F[0] = self.inject_swap(F[0], range(0, self.mid))
        # G3にrho_Aの成分を混ぜることでF0と非可換にする
        shift = random.randint(1, max(1, self.mid - 1))
        G[3] = self.compose(G[3], self.get_power(self.rho_A, shift))
        
        G[2] = self.inject_swap(G[2], range(self.mid, self.P))
        shift2 = random.randint(1, max(1, self.mid - 1))
        F[1] = self.compose(F[1], self.get_power(self.rho_B, shift2))

        return F, G

def tuple_to_sparse(p, size):
    rows = np.arange(size)
    cols = np.array(p)
    data = np.ones(size, dtype=np.int8)
    return csr_matrix((data, (rows, cols)), shape=(size, size))

def build_matrices(F_tuples, G_tuples, P, J):
    L_half = len(F_tuples)
    F_mats = [tuple_to_sparse(f, P) for f in F_tuples]
    G_mats = [tuple_to_sparse(g, P) for g in G_tuples]

    hx_rows = []
    hz_rows = []

    for i in range(J):
        hx_blocks = []
        hz_blocks = []
        for j in range(L_half):
            idx_f = (j - i) % L_half
            idx_g = (i - j) % L_half
            hx_blocks.append(F_mats[idx_f])
            hz_blocks.append(G_mats[idx_g].transpose())
        for j in range(L_half):
            idx_g = (j - i) % L_half
            idx_f = (i - j) % L_half
            hx_blocks.append(G_mats[idx_g])
            hz_blocks.append(F_mats[idx_f].transpose())
        
        hx_rows.append(hstack(hx_blocks))
        hz_rows.append(hstack(hz_blocks))

    return vstack(hx_rows), vstack(hz_rows)

In [5]:
def count_cycles_direct(H):
    num_checks, num_vars = H.shape
    adj_c = [H.getrow(i).indices for i in range(num_checks)]
    H_csc = H.tocsc()
    adj_v = [H_csc.getcol(j).indices for j in range(num_vars)]
    c4, c6 = 0, 0
    # 4-cycle
    shared = {}
    for c1 in range(num_checks):
        for v in adj_c[c1]:
            for c2 in adj_v[v]:
                if c2 > c1:
                    shared[(c1, c2)] = shared.get((c1, c2), 0) + 1
    for count in shared.values():
        if count >= 2: c4 += count * (count - 1) // 2
    # 6-cycle
    for c1 in range(num_checks):
        for v1 in adj_c[c1]:
            for c2 in adj_v[v1]:
                if c2 <= c1: continue
                for v2 in adj_c[c2]:
                    if v2 == v1: continue
                    for c3 in adj_v[v2]:
                        if c3 <= c1 or c3 == c2: continue
                        common = set(adj_c[c3]) & set(adj_c[c1])
                        for v3 in common:
                            if v3 != v1 and v3 != v2: c6 += 1
    return c4, c6 // 2

In [6]:

# 実行
optimizer = GirthOptimizer(P=768)
F_opt, G_opt = optimizer.search_optimized()

Hx, Hz = build_matrices(F_opt, G_opt, P, J)
    
# 1. まず直交性をチェック (高速)
ortho = (Hx @ Hz.transpose())
ortho.data %= 2
ortho.eliminate_zeros()
print(ortho.nnz)

# 2. サイクルをカウント
c4, c6 = count_cycles_direct(Hx)
print(f"C4: {c4}, C6: {c6}")

--- Optimized Search Starting (P=768) ---
F[0] determined. Current C4 cycles in F-set: 0
F[1] determined. Current C4 cycles in F-set: 0
F[2] determined. Current C4 cycles in F-set: 0
F[3] determined. Current C4 cycles in F-set: 0
F[4] determined. Current C4 cycles in F-set: 0
F[5] determined. Current C4 cycles in F-set: 0
G[0] determined. Current C4 cycles in Total-set: 0
G[1] determined. Current C4 cycles in Total-set: 0
G[2] determined. Current C4 cycles in Total-set: 0
G[3] determined. Current C4 cycles in Total-set: 0
G[4] determined. Current C4 cycles in Total-set: 0
G[5] determined. Current C4 cycles in Total-set: 0
0
C4: 34560, C6: 101376


In [9]:
import pandas as pd
def display_fg_commutativity_table(F, G):
    size = len(F)
    f_labels = [f"f{i}" for i in range(size)]
    g_labels = [f"g{i}" for i in range(size)]
    matrix = np.zeros((size, size), dtype=int)
    
    for i in range(size):
        for j in range(size):
            if all(F[i][G[j][x]] == G[j][F[i][x]] for x in range(len(F[i]))):
                matrix[i, j] = 1
            else:
                matrix[i, j] = 0
    
    df = pd.DataFrame(matrix, index=f_labels, columns=g_labels)
    print("\n--- F-G 間可換表 (1=可換, 0=非可換) ---")
    return df

# 実行
# F_final, G_final = opt.solve() の実行後に以下を呼び出す
commute_df = display_fg_commutativity_table(F_opt,  G_opt)
print(commute_df)


--- F-G 間可換表 (1=可換, 0=非可換) ---
    g0  g1  g2  g3  g4  g5
f0   1   1   1   1   1   1
f1   1   1   1   1   1   1
f2   1   1   1   1   1   1
f3   1   1   1   1   1   1
f4   1   1   1   1   1   1
f5   1   1   1   1   1   1
